# TRAVI — GOOGLE COLAB ON-DEMAND GPU INFERENCE BACKEND
### Qwen2-7B-Instruct with Ollama, Vietnam Tourism RAG & Adaptive Personalization

This notebook establishes an on-demand, free GPU inference server for the **Travi Vietnam Travel Platform**.
- **Hardware Target:** Google Colab T4 GPU (15GB GDDR6 VRAM)
- **Model:** `qwen2:7b` (Qwen2-7B-Instruct Q4_0 quantized GGUF, ~4.5 GB VRAM)
- **Inference Engine:** Ollama daemon with persistent VRAM pinning (`keep_alive: -1`)
- **Backend:** FastAPI with Token Guard, Semaphore Concurrency Limiter (2), and Adaptive Customer Profiling
- **Tunnel:** Cloudflare Tunnel (`pycloudflared`) exposing a temporary authenticated HTTPS endpoint.

> [!IMPORTANT]
> **Temporary Server:** This inference endpoint remains active while this Colab session is running. When the Colab runtime disconnects or restarts, the tunnel expires, and Travi seamlessly continues in Verified Deterministic Expert mode.

In [ ]:
# STEP 1: VERIFY HARDWARE & GPU AVAILABILITY
# Rule: Verify actual CUDA GPU availability before loading the model. Do not assume a GPU was assigned.
import torch
import subprocess
import sys

print("=== CHECKING GOOGLE COLAB CUDA GPU RUNTIME ===")
if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected. Please go to Colab Menu -> Runtime -> Change runtime type -> Select T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
print(f"SUCCESS: Detected {gpu_name} with {vram_mb} MB VRAM.")
subprocess.run(["nvidia-smi"])


In [ ]:
# STEP 2: INSTALL DEPENDENCIES & OLLAMA SERVER
!pip install -q fastapi uvicorn httpx pydantic pycloudflared nest-asyncio kagglehub

# Install official Ollama binary on Linux
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# STEP 3: START OLLAMA DAEMON IN BACKGROUND
# Note: Ollama binds strictly to 127.0.0.1:11434 and is NEVER exposed directly
import subprocess
import time
import httpx

print("Starting Ollama daemon on 127.0.0.1:11434...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for Ollama to become ready
for attempt in range(20):
    try:
        resp = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
        if resp.status_code == 200:
            print("Ollama daemon is healthy and ready.")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start within 20 seconds.")


In [ ]:
# STEP 5: WRITE AUDITED & HARDENED FASTAPI BACKEND APPLICATION SCRIPT (app.py)
# Hardened CORS, 16KB body size limit, prompt injection defense, and multi-collection RAG
with open("app.py", "w", encoding="utf-8") as f:
    f.write("\"\"\"\nTRAVI AI ASSISTANT — GOOGLE COLAB GPU INFERENCE BACKEND (Qwen2-7B-Instruct)\n=============================================================================\nFastAPI Service with Ollama backend, In-memory Vietnam Tourism RAG,\nAdaptive Customer Profile Personalization, Token Authentication,\nConcurrency Semaphore, and Streaming SSE support.\n\nHardware Target: Google Colab T4 GPU (15GB GDDR6 VRAM) or Tesla P100\nModel: Qwen2-7B-Instruct (Q4_0 Quantized GGUF, ~4.5GB VRAM)\n\"\"\"\n\nimport os\nimport sys\nimport json\nimport time\nimport asyncio\nimport logging\nfrom typing import List, Optional, Dict, Any\nfrom contextlib import asynccontextmanager\n\nfrom fastapi import FastAPI, Header, HTTPException, Depends, Request\nfrom fastapi.responses import StreamingResponse, JSONResponse\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel, Field\nimport httpx\n\nimport secrets\n\n# Configure logging\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\"%(asctime)s [%(levelname)s] %(name)s: %(message)s\"\n)\nlogger = logging.getLogger(\"travi-colab-gpu\")\n\n# Configuration from environment: NEVER use a predictable hardcoded default\nTOKEN_FILE_PATH = \"/content/.travi_token\" if os.path.exists(\"/content\") else \".travi_token\"\n\ndef resolve_api_token() -> str:\n    \"\"\"Safely retrieves token from env or creates a 600-permission session token file.\"\"\"\n    env_token = os.getenv(\"COLAB_AI_API_TOKEN\") or os.getenv(\"TRAVI_COLAB_API_KEY\")\n    if env_token and env_token.strip():\n        return env_token.strip()\n\n    if os.path.exists(TOKEN_FILE_PATH):\n        try:\n            with open(TOKEN_FILE_PATH, \"r\", encoding=\"utf-8\") as f:\n                token = f.read().strip()\n                if token:\n                    return token\n        except Exception:\n            pass\n\n    generated = secrets.token_urlsafe(24)\n    try:\n        with open(TOKEN_FILE_PATH, \"w\", encoding=\"utf-8\") as f:\n            f.write(generated)\n        try:\n            os.chmod(TOKEN_FILE_PATH, 0o600)\n        except Exception:\n            pass\n    except Exception:\n        pass\n    return generated\n\nCOLAB_AI_API_TOKEN = resolve_api_token()\nlogger.info(\"Authentication security initialized with cryptographically secure token.\")\n\nOLLAMA_BASE_URL = os.getenv(\"OLLAMA_BASE_URL\", \"http://127.0.0.1:11434\")\nMODEL_NAME = os.getenv(\"MODEL_NAME\", \"qwen2:7b\")\nMAX_CONCURRENT_REQUESTS = int(os.getenv(\"MAX_CONCURRENT_REQUESTS\", \"2\"))\n# Colab Ollama timeout: 35s to respect hierarchy (Ollama 35s < Gateway 45s < Cloud Functions 60s)\nREQUEST_TIMEOUT_SECONDS = float(os.getenv(\"REQUEST_TIMEOUT_SECONDS\", \"35.0\"))\nMAX_REQUEST_BODY_BYTES = 16 * 1024  # 16KB max payload limit\n\n# Concurrency limiter to protect 15GB/16GB VRAM from thrashing\ninference_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)\n\n# GPU inspection state\ngpu_info: Dict[str, Any] = {\n    \"detected\": False,\n    \"name\": \"Unknown\",\n    \"total_memory_mb\": 0,\n    \"cuda_available\": False,\n}\n\ndef verify_gpu_environment():\n    \"\"\"Verify actual CUDA GPU availability on Google Colab.\"\"\"\n    global gpu_info\n    try:\n        import torch\n        if torch.cuda.is_available():\n            gpu_count = torch.cuda.device_count()\n            device_name = torch.cuda.get_device_name(0)\n            vram_bytes = torch.cuda.get_device_properties(0).total_memory\n            vram_mb = int(vram_bytes / (1024 * 1024))\n            gpu_info = {\n                \"detected\": True,\n                \"cuda_available\": True,\n                \"device_count\": gpu_count,\n                \"name\": device_name,\n                \"total_memory_mb\": vram_mb,\n            }\n            logger.info(f\"Verified GPU: {device_name} ({vram_mb} MB VRAM)\")\n        else:\n            gpu_info = {\n                \"detected\": False,\n                \"cuda_available\": False,\n                \"name\": \"No CUDA device found\",\n                \"total_memory_mb\": 0,\n            }\n            logger.warning(\"No CUDA GPU detected! Google Colab runtime is running in CPU mode.\")\n    except Exception as e:\n        logger.error(f\"Failed to inspect GPU with PyTorch: {e}\")\n        gpu_info[\"error\"] = str(e)\n\n# In-memory Multi-Collection Vietnam Tourism RAG Corpus\nCOLLECTION_NAMES = [\n    \"tourism_pois\",\n    \"cultural_heritage\",\n    \"local_cuisine\",\n    \"festivals\",\n    \"transportation\",\n    \"accommodation\",\n    \"tourism_experiences\",\n]\n\nmulti_collections: Dict[str, List[Dict[str, Any]]] = {c: [] for c in COLLECTION_NAMES}\nrag_corpus: List[Dict[str, Any]] = []\n\ndef _extract_text_list(items: Any) -> List[str]:\n    \"\"\"Safely extracts a list of strings from either strings or structured dicts.\"\"\"\n    out = []\n    if isinstance(items, list):\n        for it in items:\n            if isinstance(it, str):\n                out.append(it)\n            elif isinstance(it, dict):\n                val = it.get(\"title\") or it.get(\"name\") or it.get(\"description\") or \"\"\n                if val:\n                    out.append(str(val))\n            elif it is not None:\n                out.append(str(it))\n    elif isinstance(items, str):\n        out.append(items)\n    return out\n\ndef load_embedded_rag_corpus():\n    \"\"\"Loads verified national multi-collections into memory for grounded retrieval.\"\"\"\n    global multi_collections, rag_corpus\n    multi_collections = {c: [] for c in COLLECTION_NAMES}\n    combined: List[Dict[str, Any]] = []\n\n    # Search directory paths for collections (supports local dev and Colab envs)\n    search_dirs = [\n        \"colab/data/collections\",\n        \"data/collections\",\n        \"src/data/national/collections\",\n        \"/content/collections\",\n        \"/content/national_collections\",\n        \"collections\",\n        \"../src/data/national/collections\",\n    ]\n\n    found_dir = None\n    for d in search_dirs:\n        if os.path.isdir(d):\n            found_dir = d\n            break\n\n    if found_dir:\n        logger.info(f\"Loading national tourism collections from directory: {found_dir}\")\n        for col in COLLECTION_NAMES:\n            file_path = os.path.join(found_dir, f\"{col}.json\")\n            if os.path.exists(file_path):\n                try:\n                    with open(file_path, \"r\", encoding=\"utf-8\") as f:\n                        data = json.load(f)\n                        if isinstance(data, list):\n                            normalized_items = []\n                            for row in data:\n                                prov = row.get(\"province_name_canonical\") or row.get(\"province\") or \"Việt Nam\"\n                                prov_id = row.get(\"province_id\")\n                                name = row.get(\"name\") or (row.get(\"localized_names\", {}).get(\"vi\")) or \"Địa điểm\"\n                                desc = row.get(\"description\") or row.get(\"short_description\") or \"\"\n\n                                exps = _extract_text_list(row.get(\"experiences\") or row.get(\"activities\"))\n                                foods = _extract_text_list(row.get(\"local_food\") or row.get(\"signature_ingredients\"))\n\n                                is_artifact = False\n                                holding_inst = None\n                                if col == \"cultural_heritage\":\n                                    h_type = row.get(\"heritage_type\")\n                                    if h_type == \"national_treasure\" or row.get(\"is_national_treasure\"):\n                                        is_artifact = True\n                                        hi = row.get(\"holding_institution\")\n                                        if isinstance(hi, dict):\n                                            holding_inst = hi.get(\"name\")\n\n                                prov_meta = row.get(\"provenance\") or {}\n                                source_name = (\n                                    prov_meta.get(\"source_name\")\n                                    if isinstance(prov_meta, dict)\n                                    else row.get(\"source\") or \"Bộ dữ liệu Du lịch Quốc gia\"\n                                )\n\n                                item = {\n                                    \"collection\": col,\n                                    \"id\": row.get(\"id\"),\n                                    \"name\": name,\n                                    \"localized_names\": row.get(\"localized_names\") or {\"vi\": name},\n                                    \"province\": prov,\n                                    \"province_id\": prov_id,\n                                    \"category\": row.get(\"category\") or col,\n                                    \"description\": desc,\n                                    \"experiences\": exps,\n                                    \"foods\": foods,\n                                    \"rituals\": _extract_text_list(row.get(\"rituals\")),\n                                    \"connected_routes\": _extract_text_list(row.get(\"connected_routes\")),\n                                    \"source\": source_name,\n                                    \"verification_status\": row.get(\"verification_status\") or \"verified\",\n                                    \"coordinate_status\": row.get(\"coordinate_confidence\") or \"none\",\n                                    \"ticket_price_notes\": row.get(\"ticket_price_notes\"),\n                                    \"opening_hours_notes\": row.get(\"opening_hours_notes\"),\n                                    \"is_artifact\": is_artifact,\n                                    \"holding_institution\": holding_inst,\n                                    \"weather_notes\": row.get(\"best_season\") or \"Tham khảo dự báo thời tiết địa phương.\"\n                                }\n                                normalized_items.append(item)\n                                combined.append(item)\n                            multi_collections[col] = normalized_items\n                            logger.info(f\"Loaded {len(normalized_items)} records into collection '{col}'\")\n                except Exception as ex:\n                    logger.warning(f\"Failed to load {file_path}: {ex}\")\n\n    # Fallback to embedded verified starter if combined is empty\n    if not combined:\n        logger.warning(\"No collection files found on disk; loading embedded verified starter corpus.\")\n        default_corpus = [\n            {\n                \"collection\": \"tourism_pois\",\n                \"id\": \"son-la-moc-chau\",\n                \"name\": \"Khu du lịch Quốc gia Mộc Châu\",\n                \"localized_names\": {\"vi\": \"Khu du lịch Quốc gia Mộc Châu\", \"en\": \"Moc Chau National Tourism Area\"},\n                \"province\": \"Sơn La\",\n                \"province_id\": \"14\",\n                \"category\": \"Khu du lịch quốc gia\",\n                \"description\": \"Cao nguyên Mộc Châu nổi tiếng với đồi chè trái tim, thung lũng mận Nà Ka, rừng thông bản Áng và thác Dải Yếm. Khí hậu mát mẻ quanh năm.\",\n                \"experiences\": [\"Ngắm hoa mận\", \"Hái chè\", \"Uống sữa tươi Mộc Châu\", \"Khám phá bản làng Thái\"],\n                \"foods\": [\"Bê chao Mộc Châu\", \"Cá suối nướng Pa Pỉnh Tộp\", \"Thịt trâu gác bếp\", \"Xôi ngũ sắc\"],\n                \"rituals\": [],\n                \"connected_routes\": [\"Quốc lộ 6\"],\n                \"source\": \"Sở Văn hóa Thể thao và Du lịch tỉnh Sơn La\",\n                \"verification_status\": \"official_gov\",\n                \"coordinate_status\": \"none\",\n                \"ticket_price_notes\": None,\n                \"opening_hours_notes\": None,\n                \"is_artifact\": False,\n                \"holding_institution\": None,\n                \"weather_notes\": \"Mùa xuân hoa mận nở trắng; mùa hè mát mẻ; mùa đông se lạnh nhiều sương mù.\"\n            },\n            {\n                \"collection\": \"cultural_heritage\",\n                \"id\": \"ca-trong-dong-ngoc-lu\",\n                \"name\": \"Trống đồng Ngọc Lũ\",\n                \"localized_names\": {\"vi\": \"Trống đồng Ngọc Lũ\", \"en\": \"Ngoc Lu Bronze Drum\"},\n                \"province\": \"Hà Nội\",\n                \"province_id\": \"01\",\n                \"category\": \"Bảo vật quốc gia\",\n                \"description\": \"Bảo vật quốc gia số 1 của Việt Nam thuộc Văn hóa Đông Sơn, niên đại thế kỷ II-III TCN. Hiện vật được lưu giữ tại Bảo tàng Lịch sử Quốc gia (Số 1 Tràng Tiền, Hà Nội).\",\n                \"experiences\": [\"Tham quan gian trưng bày Kim khí thời Hùng Vương\", \"Tìm hiểu kỹ thuật đúc đồng Đông Sơn\"],\n                \"foods\": [\"Ẩm thực phố cổ Hà Nội\", \"Chả cá Lã Vọng\", \"Cà phê trứng\"],\n                \"rituals\": [],\n                \"connected_routes\": [],\n                \"source\": \"Bảo tàng Lịch sử Quốc gia & Cục Di sản Văn hóa\",\n                \"verification_status\": \"official_gov\",\n                \"coordinate_status\": \"exact\",\n                \"ticket_price_notes\": None,\n                \"opening_hours_notes\": None,\n                \"is_artifact\": True,\n                \"holding_institution\": \"Bảo tàng Lịch sử Quốc gia\",\n                \"weather_notes\": \"Bảo tàng mở cửa quanh năm trong phòng trưng bày điều hòa.\"\n            },\n            {\n                \"collection\": \"cultural_heritage\",\n                \"id\": \"ca-nghe-thuat-xoe-thai\",\n                \"name\": \"Nghệ thuật Xòe Thái\",\n                \"localized_names\": {\"vi\": \"Nghệ thuật Xòe Thái\", \"en\": \"Art of Xoe Dance of the Tai People\"},\n                \"province\": \"Sơn La\",\n                \"province_id\": \"14\",\n                \"category\": \"Di sản Văn hóa Phi vật thể UNESCO\",\n                \"description\": \"Điệu múa dân gian truyền thống biểu trưng cho tinh thần cộng đồng, lòng hiếu khách của người Thái Tây Bắc.\",\n                \"experiences\": [\"Tham gia vòng xòe hoa tại Bản Áng\", \"Giao lưu văn nghệ dân gian\"],\n                \"foods\": [\"Cơm lam\", \"Cá nướng Pa Pỉnh Tộp\", \"Rượu cần\"],\n                \"rituals\": [\"Múa xòe vòng\", \"Uống rượu cần\"],\n                \"connected_routes\": [],\n                \"source\": \"UNESCO & Sở VHTT&DL Sơn La\",\n                \"verification_status\": \"official_gov\",\n                \"coordinate_status\": \"none\",\n                \"ticket_price_notes\": None,\n                \"opening_hours_notes\": None,\n                \"is_artifact\": False,\n                \"holding_institution\": None,\n                \"weather_notes\": \"Các lễ hội diễn ra rộn ràng vào dịp đầu xuân và mùa gặt.\"\n            },\n            {\n                \"collection\": \"local_cuisine\",\n                \"id\": \"dish-be-chao-moc-chau\",\n                \"name\": \"Bê chao Mộc Châu\",\n                \"localized_names\": {\"vi\": \"Bê chao Mộc Châu\", \"en\": \"Moc Chau Crisp-Fried Veal\"},\n                \"province\": \"Sơn La\",\n                \"province_id\": \"14\",\n                \"category\": \"Đặc sản địa phương\",\n                \"description\": \"Thịt bê đồi non được chao nhanh trong chảo dầu sôi già. Bì giòn, thịt mềm ngọt mọng nước chấm cùng tương bần đậm đà.\",\n                \"experiences\": [\"Thưởng thức tại trang trại thảo nguyên Mộc Châu\"],\n                \"foods\": [\"Bê non Mộc Châu\", \"Tương bần\", \"Gừng sả\"],\n                \"rituals\": [],\n                \"connected_routes\": [],\n                \"source\": \"Sở VHTT&DL tỉnh Sơn La\",\n                \"verification_status\": \"official_gov\",\n                \"coordinate_status\": \"approximate\",\n                \"ticket_price_notes\": None,\n                \"opening_hours_notes\": None,\n                \"is_artifact\": False,\n                \"holding_institution\": None,\n                \"weather_notes\": \"Quanh năm, đặc biệt ngon vào mùa se lạnh.\"\n            },\n            {\n                \"collection\": \"tourism_experiences\",\n                \"id\": \"exp-san-may-ta-xua\",\n                \"name\": \"Săn mây Tà Xùa & Sống lưng Khủng long\",\n                \"localized_names\": {\"vi\": \"Săn mây Tà Xùa\", \"en\": \"Ta Xua Cloud Hunting\"},\n                \"province\": \"Sơn La\",\n                \"province_id\": \"14\",\n                \"category\": \"Trải nghiệm Khám phá\",\n                \"description\": \"Đón bình minh giữa biển mây bồng bềnh tại xã Tà Xùa và sống lưng khủng long Háng Đồng cheo leo trên độ cao 2.000m.\",\n                \"experiences\": [\"Săn mây Tà Xùa\", \"Trekking sống lưng khủng long\", \"Thưởng thức chè Shan tuyết\"],\n                \"foods\": [\"Thịt lợn bản nướng\", \"Măng trúc xào\", \"Cơm lam\"],\n                \"rituals\": [],\n                \"connected_routes\": [\"Tỉnh lộ 112\"],\n                \"source\": \"Sở Văn hóa Thể thao và Du lịch tỉnh Sơn La\",\n                \"verification_status\": \"official_gov\",\n                \"coordinate_status\": \"approximate\",\n                \"ticket_price_notes\": None,\n                \"opening_hours_notes\": None,\n                \"is_artifact\": False,\n                \"holding_institution\": None,\n                \"weather_notes\": \"Mùa săn mây đẹp nhất từ tháng 10 đến tháng 4 năm sau.\"\n            }\n        ]\n        combined = default_corpus\n        for it in default_corpus:\n            multi_collections[it[\"collection\"]].append(it)\n\n    rag_corpus = combined\n    logger.info(f\"Successfully loaded {len(rag_corpus)} verified facts across {len(multi_collections)} national collections.\")\n\ndef retrieve_rag_context(\n    query: str,\n    top_k: int = 4,\n    collections: Optional[List[str]] = None,\n    provinces: Optional[List[str]] = None\n) -> List[Dict[str, Any]]:\n    \"\"\"\n    Source-Aware BM25/Token-Ranked Retrieval with Multi-Collection & Administrative Hierarchy Filtering.\n    Strictly token-based with source authority weighting: official_gov (1.5x) > verified (1.3x) > needs_review (1.0x) > unverified (0.7x).\n    Honest retrieval disclosure: Zero vector embeddings claim.\n    \"\"\"\n    import re\n    q_lower = query.lower().strip()\n    if not q_lower:\n        return []\n\n    q_tokens = [t for t in re.sub(r\"[^\\w\\s]\", \" \", q_lower).split() if len(t) > 1]\n\n    target_corpus = rag_corpus\n    if collections:\n        cols_set = set(collections)\n        target_corpus = [item for item in target_corpus if item.get(\"collection\") in cols_set]\n\n    if provinces:\n        provs_lower = [p.lower().strip() for p in provinces]\n        target_corpus = [\n            item for item in target_corpus\n            if any(\n                p in item.get(\"province\", \"\").lower()\n                or (item.get(\"province_id\") and p == item.get(\"province_id\", \"\").lower())\n                for p in provs_lower\n            )\n        ]\n\n    scored = []\n    for item in target_corpus:\n        raw_score = 0\n        name = item.get(\"name\", \"\").lower()\n        desc = item.get(\"description\", \"\").lower()\n        prov = item.get(\"province\", \"\").lower()\n        exps = \" \".join(item.get(\"experiences\", [])).lower()\n        foods = \" \".join(item.get(\"foods\", [])).lower()\n        routes = \" \".join(item.get(\"connected_routes\", [])).lower()\n        searchable_text = f\"{name} {desc} {prov} {exps} {foods} {routes}\"\n\n        # 1. Exact match on title\n        if q_lower == name:\n            raw_score += 50\n        elif name in q_lower or q_lower in name:\n            raw_score += 25\n\n        # 2. Province match\n        if prov and prov in q_lower:\n            raw_score += 15\n\n        # 3. Token match\n        for token in q_tokens:\n            if token in name:\n                raw_score += 8\n            elif token in searchable_text:\n                raw_score += 2\n\n        if raw_score > 0:\n            status = item.get(\"verification_status\", \"unverified\")\n            if status == \"official_gov\":\n                multiplier = 1.5\n            elif status == \"verified\":\n                multiplier = 1.3\n            elif status == \"needs_review\":\n                multiplier = 1.0\n            else:\n                multiplier = 0.7\n\n            final_score = raw_score * multiplier\n            scored.append((final_score, item))\n\n    scored.sort(key=lambda x: x[0], reverse=True)\n\n    # Deduplicate results\n    seen_ids = set()\n    results = []\n    for score, item in scored:\n        item_id = item.get(\"id\")\n        if item_id and item_id in seen_ids:\n            continue\n        seen_ids.add(item_id)\n        item_copy = dict(item)\n        item_copy[\"score\"] = round(score, 1)\n        results.append(item_copy)\n        if len(results) >= top_k:\n            break\n\n    return results\n\n# Lifespan: warm up model in Ollama\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    verify_gpu_environment()\n    load_embedded_rag_corpus()\n    # Check Ollama connectivity & keep model loaded in VRAM\n    async with httpx.AsyncClient(timeout=15.0) as client:\n        try:\n            resp = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if resp.status_code == 200:\n                logger.info(f\"Ollama server connected at {OLLAMA_BASE_URL}.\")\n                warmup_payload = {\n                    \"model\": MODEL_NAME,\n                    \"prompt\": \"Xin chào, trợ lý du lịch Việt Nam đã sẵn sàng.\",\n                    \"keep_alive\": -1,\n                    \"stream\": False\n                }\n                logger.info(f\"Warming up {MODEL_NAME} with keep_alive: -1...\")\n                warmup_resp = await client.post(f\"{OLLAMA_BASE_URL}/api/generate\", json=warmup_payload, timeout=60.0)\n                if warmup_resp.status_code == 200:\n                    logger.info(f\"Model {MODEL_NAME} successfully loaded and pinned in VRAM.\")\n            else:\n                logger.warning(f\"Ollama responded with status {resp.status_code}\")\n        except Exception as e:\n            logger.error(f\"Could not connect to Ollama: {e}. Ensure Ollama is running.\")\n    yield\n    logger.info(\"FastAPI Colab service shutting down.\")\n\napp = FastAPI(\n    title=\"Travi AI Assistant — Google Colab GPU Backend\",\n    description=\"On-demand GPU inference runtime serving Qwen2-7B-Instruct with Vietnam Tourism RAG & Adaptive Personalization\",\n    version=\"1.2.0\",\n    lifespan=lifespan\n)\n\n# 16KB Request Payload Guard Middleware\n@app.middleware(\"http\")\nasync def enforce_payload_limit(request: Request, call_next):\n    content_length = request.headers.get(\"content-length\")\n    if content_length:\n        try:\n            if int(content_length) > MAX_REQUEST_BODY_BYTES:\n                return JSONResponse(\n                    status_code=413,\n                    content={\"error\": \"Payload Too Large: Maximum allowed request size is 16KB.\", \"code\": \"PAYLOAD_TOO_LARGE\"}\n                )\n        except ValueError:\n            pass\n\n    # Read body safely to check chunked requests\n    body = await request.body()\n    if len(body) > MAX_REQUEST_BODY_BYTES:\n        return JSONResponse(\n            status_code=413,\n            content={\"error\": \"Payload Too Large: Maximum allowed request size is 16KB.\", \"code\": \"PAYLOAD_TOO_LARGE\"}\n        )\n\n    return await call_next(request)\n\n# Hardened CORS configuration (Zero wildcard * with credentials)\nALLOWED_ORIGINS = [\n    \"http://localhost:5173\",\n    \"http://127.0.0.1:5173\",\n    \"http://localhost:3000\",\n    \"http://localhost:3001\",\n    \"https://travi-app-23th10.web.app\",\n    \"https://travi-app-23th10.firebaseapp.com\",\n]\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=ALLOWED_ORIGINS,\n    allow_credentials=False,\n    allow_methods=[\"GET\", \"POST\", \"OPTIONS\"],\n    allow_headers=[\"*\"],\n)\n\n# Authentication Dependency with Constant-Time Verification\ndef verify_token(\n    authorization: Optional[str] = Header(None),\n    x_api_key: Optional[str] = Header(None)\n):\n    token = None\n    if authorization and authorization.startswith(\"Bearer \"):\n        token = authorization.replace(\"Bearer \", \"\").strip()\n    elif x_api_key:\n        token = x_api_key.strip()\n\n    if not token or not secrets.compare_digest(token, COLAB_AI_API_TOKEN):\n        raise HTTPException(\n            status_code=401,\n            detail=\"Unauthorized: Invalid or missing API authentication token.\",\n            headers={\"WWW-Authenticate\": \"Bearer\"}\n        )\n    return token\n\n# Request & Response Schemas\nclass LocationInfo(BaseModel):\n    country: Optional[str] = \"Việt Nam\"\n    province: Optional[str] = None\n    city: Optional[str] = None\n\nclass CustomerProfile(BaseModel):\n    # Distinct coexisting dimensions:\n    market: Optional[str] = Field(default=None, description=\"DOMESTIC | INTERNATIONAL\")\n    origin_location: Optional[LocationInfo] = None\n    destination_location: Optional[LocationInfo] = None\n    travel_scope: Optional[str] = Field(default=None, description=\"LOCAL | INTERPROVINCIAL | INTERNATIONAL_TRAVEL\")\n    persona: Optional[str] = Field(default=None, description=\"FAMILY | BACKPACKER | SOLO | EXPLORER\")\n    language: Optional[str] = Field(default=None, description=\"VI | EN | OTHER\")\n\n    # Legacy & convenience fields\n    customer_type: Optional[str] = Field(default=None, description=\"Legacy alias: INTERNATIONAL | DOMESTIC | LOCAL | INTERPROVINCIAL\")\n    origin_country: Optional[str] = Field(default=\"VN\")\n    origin_province: Optional[str] = None\n    destination: Optional[str] = None\n    preferred_language: Optional[str] = Field(default=\"vi\")\n    budget: Optional[str] = Field(default=\"moderate\")\n    travel_duration: Optional[int] = Field(default=3)\n    interests: Optional[List[str]] = Field(default=[])\n    transportation_preference: Optional[str] = None\n\nclass ChatMessage(BaseModel):\n    role: str = Field(..., description=\"'user', 'assistant', or 'system'\")\n    content: str = Field(..., min_length=1, max_length=1000)\n\nclass ChatRequest(BaseModel):\n    prompt: str = Field(..., min_length=1, max_length=1000, description=\"User travel query\")\n    profile: Optional[CustomerProfile] = None\n    context: Optional[Dict[str, Any]] = None\n    history: Optional[List[ChatMessage]] = Field(default=[], max_length=10, description=\"Multi-turn conversation history (max 10 turns)\")\n    stream: Optional[bool] = Field(default=False, description=\"Enable Server-Sent Events streaming\")\n    # Prompt injection defense: client-side system overrides are rejected / ignored\n\nclass ModelInfoResponse(BaseModel):\n    model: str\n    quantization: str\n    gpu: Dict[str, Any]\n    keep_alive: str\n    rag_facts_count: int\n    context_window: int\n    runtime_expires_note: str\n    active_model_ready: bool = True\n    concurrency_max: int = MAX_CONCURRENT_REQUESTS\n    collections_summary: Optional[Dict[str, int]] = None\n\ndef build_adaptive_system_prompt(profile: Optional[CustomerProfile], rag_facts_text: str) -> str:\n    \"\"\"\n    Constructs prompt customized to multi-dimensional customer attributes:\n    - Market: DOMESTIC / INTERNATIONAL\n    - Travel Scope: LOCAL / INTERPROVINCIAL / INTERNATIONAL_TRAVEL\n    - Persona: FAMILY / BACKPACKER / SOLO / EXPLORER\n    - Language: VI / EN / OTHER\n    All dimensions coexist without conflicting or stereotyping.\n    \"\"\"\n    is_english = False\n    if profile:\n        if profile.language and profile.language.upper() == \"EN\":\n            is_english = True\n        elif profile.preferred_language and profile.preferred_language.lower().startswith(\"en\"):\n            is_english = True\n\n    if is_english:\n        base_instructions = (\n            \"You are Travi AI — a dedicated Vietnam travel and cultural intelligence assistant.\\n\"\n            \"DATA INTEGRITY & ETHICAL TOURISM PRINCIPLES (MANDATORY):\\n\"\n            \"1. ONLY state verified facts. NEVER fabricate ticket prices, opening hours, or assumed coordinates.\\n\"\n            \"2. Distinguish clearly: Cultural artifacts kept in museums differ from archaeological discovery sites; artifact coordinates are not museum entrance gates.\\n\"\n            \"3. If a site's coordinates or status have unverified field notes, advise travelers to confirm with local management before traveling.\\n\"\n            \"4. If ticket prices, opening hours, or phone numbers are absent in the retrieved facts, you MUST state: 'Thông tin chưa được xác thực, vui lòng liên hệ ban quản lý địa phương.' (or 'Official information is not yet verified, please contact local site management.'). NEVER guess or estimate them.\\n\"\n            \"5. Respond in polite, engaging, and culturally evocative English while preserving authentic Vietnamese place and dish names.\"\n        )\n    else:\n        base_instructions = (\n            \"Bạn là Travi AI — Trợ lý chuyên gia du lịch & văn hóa Việt Nam tận tâm.\\n\"\n            \"NGUYÊN TẮC TOÀN VẸN DỮ LIỆU & ĐẠO ĐỨC DU LỊCH (BẮT BUỘC TUÂN THỦ):\\n\"\n            \"1. CHỈ sử dụng sự thật đã xác minh. TUYỆT ĐỐI KHÔNG tự tạo ra giá vé, giờ mở cửa hoặc tọa độ giả định.\\n\"\n            \"2. Phân biệt rõ: Hiện vật văn hóa (Bảo tàng lưu giữ) KHÁC với địa điểm khảo cổ nơi phát hiện; tọa độ hiện vật không phải cổng bảo tàng.\\n\"\n            \"3. Nếu thông tin hoặc tọa độ của điểm đến chưa được đối soát thực địa, hãy nhắc nhở du khách liên hệ ban quản lý địa phương trước khi di chuyển.\\n\"\n            \"4. Nếu giá vé, giờ mở cửa hoặc số điện thoại không có trong thông tin tra cứu, BẮT BUỘC nêu rõ: 'Thông tin chưa được xác thực, vui lòng liên hệ ban quản lý địa phương.' TUYỆT ĐỐI KHÔNG tự bịa hoặc ước lượng.\\n\"\n            \"5. Trả lời bằng tiếng Việt lịch sự, truyền cảm, có chiều sâu văn hóa bản địa.\"\n        )\n\n    guidance_sections = []\n\n    # 1. Market Dimension (Domestic vs International)\n    market = \"DOMESTIC\"\n    if profile:\n        if profile.market:\n            market = profile.market.upper()\n        elif profile.customer_type == \"INTERNATIONAL\" or (profile.origin_country and profile.origin_country.lower() not in [\"vn\", \"vietnam\", \"việt nam\"]):\n            market = \"INTERNATIONAL\"\n\n    if market == \"INTERNATIONAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [International Market]: Provide deep cultural context, explain local customs, \"\n                \"temple & pagoda etiquette, VND currency conversion, e-visa/sim card advice, and food safety.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Thị trường Quốc tế]: Cung cấp hướng dẫn văn hóa sâu sắc, giải thích phong tục tập quán, \"\n                \"nghi thức đền chùa, quy đổi tiền tệ VND, sim/data và an toàn thực phẩm.\"\n            )\n    else:\n        if is_english:\n            guidance_sections.append(\n                \"- [Domestic Market]: Emphasize authentic local cuisine, cost-effective itineraries, and prominent landmarks.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Thị trường Nội địa]: Chú trọng ẩm thực chuẩn vị địa phương, chi phí hợp lý và địa điểm nổi tiếng.\"\n            )\n\n    # 2. Travel Scope Dimension (Local vs Interprovincial vs Cross-Border)\n    scope = None\n    if profile:\n        scope = profile.travel_scope\n        if not scope and profile.customer_type:\n            if profile.customer_type in [\"LOCAL\", \"INTERPROVINCIAL\", \"INTERNATIONAL_TRAVEL\"]:\n                scope = profile.customer_type\n\n    if scope == \"LOCAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Local Scope]: Focus on quiet hidden gems, humble local eateries, avoiding overcrowded tourist traps; suggest weekend getaways.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Phạm vi Bản địa]: Tập trung vào điểm đến ngách (hidden gems), quán ăn dân dã ít người biết, \"\n                \"tránh tụ điểm đông đúc du lịch đại trà; gợi ý dã ngoại ngắn cuối tuần.\"\n            )\n    elif scope == \"INTERPROVINCIAL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Interprovincial Scope]: Optimize transportation routes, mountain pass conditions, travel duration, \"\n                \"and cultural connections between adjoining provinces.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Phạm vi Liên tỉnh]: Tối ưu hóa cung đường di chuyển, cảnh báo đèo dốc và thời gian trung chuyển; \"\n                \"kết nối các trải nghiệm văn hóa giữa các tỉnh thành liên quan.\"\n            )\n    elif scope == \"INTERNATIONAL_TRAVEL\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Cross-Border Scope]: Assist with international flight connections, airport transfers, and customs guidance.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Phạm vi Xuyên quốc gia]: Hỗ trợ thông tin thị thực/e-visa, kết nối chuyến bay và trung chuyển sân bay.\"\n            )\n\n    # 3. Persona Dimension (Family / Backpacker / Solo / Explorer)\n    persona = profile.persona if profile else None\n    if persona == \"FAMILY\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Family Persona]: Prioritize safe, accessible attractions for children and elderly travelers, relaxing pacing, and cozy dining.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Chân dung Gia đình]: Ưu tiên các điểm tham quan an toàn, dễ tiếp cận cho người lớn tuổi và trẻ nhỏ, \"\n                \"nhà hàng có không gian ấm cúng, lịch trình thong thả tránh quá tải.\"\n            )\n    elif persona == \"BACKPACKER\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Backpacker Persona]: Highlight winding mountain passes, authentic homestays, street food gems, budget optimization, and trekking.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Chân dung Phượt thủ / Bụi]: Ưu tiên trải nghiệm cung đường đèo dốc, homestay bản địa, ẩm thực đường phố ngách \"\n                \"và các hoạt động khám phá mạo hiểm vừa sức, tối ưu chi phí.\"\n            )\n    elif persona == \"SOLO\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Solo Persona]: Suggest peaceful, secure spaces, scenic coffee spots, personal rejuvenation, and mindful cultural immersion.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Chân dung Độc hành (Solo)]: Gợi ý không gian yên tĩnh, an toàn, cà phê ngắm cảnh và trải nghiệm chữa lành, chiêm nghiệm văn hóa.\"\n            )\n    elif persona == \"EXPLORER\":\n        if is_english:\n            guidance_sections.append(\n                \"- [Explorer Persona]: Prioritize deep cultural heritage, traditional handicraft villages, archaeological history, and indigenous roots.\"\n            )\n        else:\n            guidance_sections.append(\n                \"- [Chân dung Nhà khám phá]: Ưu tiên chiều sâu di sản, làng nghề thủ công cổ truyền, lịch sử khảo cổ và nguồn gốc ẩm thực.\"\n            )\n\n    full_system = base_instructions\n    header_guidance = \"### ADAPTIVE CONSULTATION GUIDELINES (COEXISTING PROFILES):\" if is_english else \"### ĐỊNH HƯỚNG TƯ VẤN THÍCH ỨNG (COEXISTING PROFILES):\"\n    if guidance_sections:\n        full_system += f\"\\n\\n{header_guidance}\\n\" + \"\\n\".join(guidance_sections)\n\n    if profile:\n        details = []\n        origin_text = profile.origin_province or (profile.origin_location.province if profile.origin_location else None)\n        dest_text = profile.destination or (profile.destination_location.province if profile.destination_location else None)\n        if dest_text: details.append(f\"Destination: {dest_text}\" if is_english else f\"Điểm đến: {dest_text}\")\n        if origin_text: details.append(f\"Origin: {origin_text}\" if is_english else f\"Khởi hành: {origin_text}\")\n        if profile.budget: details.append(f\"Budget: {profile.budget}\" if is_english else f\"Ngân sách: {profile.budget}\")\n        if profile.travel_duration: details.append(f\"Duration: {profile.travel_duration} days\" if is_english else f\"Thời gian: {profile.travel_duration} ngày\")\n        if profile.interests: details.append(f\"Interests: {', '.join(profile.interests)}\" if is_english else f\"Sở thích: {', '.join(profile.interests)}\")\n        if details:\n            header_profile = \"### TRIP PROFILE INFORMATION:\" if is_english else \"### THÔNG TIN HỒ SƠ HÀNH TRÌNH:\"\n            full_system += f\"\\n\\n{header_profile}\\n- \" + \"\\n- \".join(details)\n\n    if rag_facts_text:\n        header_rag = \"### VERIFIED RETRIEVED FACTS (RAG GROUNDING):\" if is_english else \"### THÔNG TIN TRA CỨU ĐÃ ĐỐI SOÁT (RETRIEVED FACTS):\"\n        full_system += f\"\\n\\n{header_rag}\\n{rag_facts_text}\"\n\n    return full_system\n\n# Endpoints\n@app.get(\"/health\")\nasync def health_check():\n    \"\"\"\n    Public health check endpoint returning minimal non-sensitive data.\n    Never exposes GPU diagnostics, VRAM, model inventory, or internal concurrency counter without auth.\n    \"\"\"\n    ollama_ok = False\n    model_ready = False\n    try:\n        async with httpx.AsyncClient(timeout=3.0) as client:\n            res = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if res.status_code == 200:\n                ollama_ok = True\n                models = [m.get(\"name\") for m in res.json().get(\"models\", [])]\n                model_ready = MODEL_NAME in models or any(MODEL_NAME in m for m in models)\n    except Exception:\n        pass\n\n    return {\n        \"status\": \"healthy\" if (ollama_ok and model_ready) else \"degraded\" if ollama_ok else \"offline\",\n        \"service\": \"travi-colab-gpu-inference\",\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime())\n    }\n\n@app.get(\"/api/model-info\", response_model=ModelInfoResponse)\nasync def model_info(token: str = Depends(verify_token)):\n    \"\"\"Provides model metadata, quantization details and memory footprint (Protected).\"\"\"\n    ollama_ok = False\n    model_ready = False\n    try:\n        async with httpx.AsyncClient(timeout=3.0) as client:\n            res = await client.get(f\"{OLLAMA_BASE_URL}/api/tags\")\n            if res.status_code == 200:\n                ollama_ok = True\n                models = [m.get(\"name\") for m in res.json().get(\"models\", [])]\n                model_ready = MODEL_NAME in models or any(MODEL_NAME in m for m in models)\n    except Exception:\n        pass\n\n    return ModelInfoResponse(\n        model=MODEL_NAME,\n        quantization=\"Q4_0 (GGUF 4-bit Quantization)\",\n        gpu=gpu_info,\n        keep_alive=\"infinite (-1, model persists in VRAM)\",\n        rag_facts_count=len(rag_corpus),\n        context_window=32768,\n        runtime_expires_note=\"Temporary Google Colab runtime. Endpoint expires when notebook kernel disconnects.\",\n        active_model_ready=model_ready,\n        concurrency_max=MAX_CONCURRENT_REQUESTS,\n        collections_summary={k: len(v) for k, v in multi_collections.items()}\n    )\n\n@app.post(\"/api/chat\")\nasync def chat(request: ChatRequest, token: str = Depends(verify_token)):\n    \"\"\"\n    Main travel AI inference endpoint with Vietnam Tourism Multi-Collection RAG grounding,\n    Adaptive Customer Profile Personalization, and concurrency locking.\n    \"\"\"\n    # 1. Retrieve RAG grounding facts with multi-collection and province support\n    req_collections = None\n    req_provinces = None\n    if request.context:\n        if isinstance(request.context.get(\"collections\"), list):\n            req_collections = request.context.get(\"collections\")\n        if isinstance(request.context.get(\"provinces\"), list):\n            req_provinces = request.context.get(\"provinces\")\n\n    retrieved_facts = retrieve_rag_context(\n        request.prompt,\n        top_k=4,\n        collections=req_collections,\n        provinces=req_provinces\n    )\n    rag_context_text = \"\"\n    sources_metadata = []\n\n    if retrieved_facts:\n        for fact in retrieved_facts:\n            is_art = fact.get(\"is_artifact\", False)\n            art_note = f\" (Hiện vật bảo tàng lưu giữ tại {fact.get('holding_institution')})\" if is_art else \"\"\n            badge = \"[CHÍNH THỐNG]\" if fact.get(\"verification_status\") == \"official_gov\" else \"[ĐÃ XÁC MINH]\" if fact.get(\"verification_status\") == \"verified\" else \"[CẦN ĐỐI SOÁT]\"\n            coord_note = \" (Tọa độ chưa đối soát thực địa)\" if fact.get(\"coordinate_status\") == \"none\" else \"\"\n            price_note = fact.get(\"ticket_price_notes\") or \"Chưa có thông tin chính thức (liên hệ ban quản lý)\"\n            hours_note = fact.get(\"opening_hours_notes\") or \"Chưa có thông tin chính thức (liên hệ ban quản lý)\"\n\n            rag_context_text += (\n                f\"- [{fact.get('collection', 'tourism')}] {fact['name']} ({fact['province']}){art_note} {badge}\\n\"\n                f\"  Mô tả: {fact['description']}\\n\"\n                f\"  Trải nghiệm: {', '.join(fact.get('experiences', []))}\\n\"\n                f\"  Ẩm thực: {', '.join(fact.get('foods', []))}\\n\"\n                f\"  Giá vé tham khảo: {price_note}\\n\"\n                f\"  Giờ mở cửa: {hours_note}\\n\"\n                f\"  Tọa độ / Điểm đến: {fact.get('coordinate_status', 'none')}{coord_note}\\n\"\n                f\"  Thời tiết / Mùa đẹp: {fact.get('weather_notes', '')}\\n\"\n                f\"  Nguồn đối soát: {fact['source']}\\n\"\n            )\n            sources_metadata.append({\n                \"name\": fact[\"name\"],\n                \"province\": fact[\"province\"],\n                \"source\": fact[\"source\"],\n                \"collection\": fact.get(\"collection\"),\n                \"verification_status\": fact.get(\"verification_status\", \"unverified\"),\n                \"coordinate_status\": fact.get(\"coordinate_status\", \"unverified\"),\n                \"is_artifact\": is_art,\n                \"ticket_price_notes\": fact.get(\"ticket_price_notes\"),\n                \"opening_hours_notes\": fact.get(\"opening_hours_notes\"),\n            })\n\n    # 2. Build personalized system prompt (Server controlled - Prompt Injection Defense)\n    system_prompt = build_adaptive_system_prompt(request.profile, rag_context_text)\n\n    # 3. Build Ollama messages array\n    messages = [{\"role\": \"system\", \"content\": system_prompt}]\n\n    if request.history:\n        for prev in request.history[-10:]:\n            messages.append({\"role\": prev.role, \"content\": prev.content})\n\n    messages.append({\"role\": \"user\", \"content\": request.prompt})\n\n    # 4. Handle Streaming SSE vs Non-Streaming JSON\n    cust_type = (request.profile.customer_type if request.profile and request.profile.customer_type else \"DOMESTIC\")\n    if request.stream:\n        return StreamingResponse(\n            stream_generator(messages, sources_metadata, cust_type),\n            media_type=\"text/event-stream\"\n        )\n    else:\n        return await non_streaming_generate(messages, sources_metadata, cust_type)\n\nasync def non_streaming_generate(messages: List[Dict[str, str]], sources: List[Dict[str, Any]], customer_type: str):\n    \"\"\"Executes non-streaming completion with concurrency lock and timeout.\"\"\"\n    try:\n        async with asyncio.timeout(REQUEST_TIMEOUT_SECONDS):\n            async with inference_semaphore:\n                async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT_SECONDS) as client:\n                    payload = {\n                        \"model\": MODEL_NAME,\n                        \"messages\": messages,\n                        \"stream\": False,\n                        \"keep_alive\": -1,\n                        \"options\": {\n                            \"temperature\": 0.6,\n                            \"top_p\": 0.9,\n                            \"num_ctx\": 4096\n                        }\n                    }\n                    response = await client.post(f\"{OLLAMA_BASE_URL}/api/chat\", json=payload)\n                    if response.status_code != 200:\n                        raise HTTPException(\n                            status_code=502,\n                            detail=f\"Ollama upstream error: {response.text}\"\n                        )\n                    data = response.json()\n                    content = data.get(\"message\", {}).get(\"content\", \"\")\n\n                    return {\n                        \"status\": \"success\",\n                        \"content\": content,\n                        \"reply\": content,\n                        \"sources\": sources,\n                        \"personalized_for\": customer_type,\n                        \"suggestions\": [\n                            \"Tìm hiểu ẩm thực bản địa\",\n                            \"Phương tiện di chuyển đến đây\",\n                            \"Thời điểm lý tưởng nhất trong năm\"\n                        ],\n                        \"provenance\": {\n                            \"providerName\": f\"Google Colab GPU Qwen2-7B ({gpu_info.get('name', 'T4')})\",\n                            \"isLiveAi\": True,\n                            \"modeDescription\": \"Ollama Serverless GPU Container via Secure HTTPS Tunnel\",\n                            \"vramAllocatedMB\": gpu_info.get(\"total_memory_mb\", 15360)\n                        }\n                    }\n    except asyncio.TimeoutError:\n        raise HTTPException(\n            status_code=504,\n            detail=f\"Inference timed out after {REQUEST_TIMEOUT_SECONDS}s.\"\n        )\n    except Exception as e:\n        logger.error(f\"Inference execution failed: {e}\")\n        raise HTTPException(status_code=500, detail=str(e))\n\nasync def stream_generator(messages: List[Dict[str, str]], sources: List[Dict[str, Any]], customer_type: str):\n    \"\"\"Yields Server-Sent Events (SSE) token chunks as generated by Ollama.\"\"\"\n    try:\n        async with inference_semaphore:\n            async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT_SECONDS) as client:\n                payload = {\n                    \"model\": MODEL_NAME,\n                    \"messages\": messages,\n                    \"stream\": True,\n                    \"keep_alive\": -1,\n                    \"options\": {\n                        \"temperature\": 0.6,\n                        \"top_p\": 0.9,\n                        \"num_ctx\": 4096\n                    }\n                }\n                async with client.stream(\"POST\", f\"{OLLAMA_BASE_URL}/api/chat\", json=payload) as resp:\n                    if resp.status_code != 200:\n                        yield f\"data: {json.dumps({'error': 'Upstream Ollama streaming failed'})}\\n\\n\"\n                        return\n\n                    async for line in resp.aiter_lines():\n                        if not line:\n                            continue\n                        try:\n                            chunk = json.loads(line)\n                            token = chunk.get(\"message\", {}).get(\"content\", \"\")\n                            done = chunk.get(\"done\", False)\n                            event_data = {\n                                \"chunk\": token,\n                                \"done\": done\n                            }\n                            if done:\n                                event_data[\"sources\"] = sources\n                                event_data[\"personalized_for\"] = customer_type\n                                event_data[\"provenance\"] = {\n                                    \"providerName\": f\"Google Colab GPU Qwen2-7B ({gpu_info.get('name', 'T4')})\",\n                                    \"isLiveAi\": True\n                                }\n                            yield f\"data: {json.dumps(event_data)}\\n\\n\"\n                            if done:\n                                break\n                        except Exception:\n                            continue\n    except Exception as err:\n        yield f\"data: {json.dumps({'error': str(err)})}\\n\\n\"\n\nif __name__ == \"__main__\":\n    import uvicorn\n    port = int(os.getenv(\"PORT\", \"8000\"))\n    logger.info(f\"Starting Travi Colab GPU Backend on 127.0.0.1:{port}...\")\n    uvicorn.run(app, host=\"127.0.0.1\", port=port, log_level=\"info\")\n\n")
print("Saved audited app.py with Multi-Collection RAG Grounding, Adaptive Profiles, 16KB Guard, and Token Security.")


In [ ]:
# STEP 6: START FASTAPI & LAUNCH SECURE CLOUDFLARE TUNNEL
import threading
import uvicorn
from pycloudflared import try_cloudflare
import app as colab_app

def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="warning")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

# Launch temporary Cloudflare HTTPS tunnel to expose 127.0.0.1:8000
tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.strip()
session_token = colab_app.COLAB_AI_API_TOKEN

print("=" * 75)
print("🚀 TRAVI GOOGLE COLAB GPU INFERENCE BACKEND IS ONLINE!")
print(f"Public HTTPS Endpoint: {public_url}")
print(f"API Token saved to: {colab_app.TOKEN_FILE_PATH} (mode 0600)")
print(f"Health Check URL (Public): {public_url}/health")
print(f"Model Info URL (Auth Required): {public_url}/api/model-info")
print(f"Chat Endpoint (Auth Required): {public_url}/api/chat")
print("=" * 75)
print("CONFIGURING YOUR TRAVI SERVER (.env):")
print(f"COLAB_AI_BASE_URL={public_url}")
print(f"COLAB_AI_API_TOKEN={session_token}")
print("COLAB_AI_ENABLED=true")
print("=" * 75)


In [ ]:
# STEP 7: AUTOMATED VERIFICATION SELF-TESTS
import httpx
headers = {"Authorization": f"Bearer {colab_app.COLAB_AI_API_TOKEN}"}
client = httpx.Client(base_url="http://127.0.0.1:8000", headers=headers, timeout=35.0)

# 1. Health check (Public minimal non-sensitive data)
h = httpx.get("http://127.0.0.1:8000/health").json()
print("1. Health Check:", h["status"], "Service:", h["service"])
assert h["status"] in ["healthy", "degraded"]
assert "gpu" not in h, "Security failure: Public health check leaked GPU data"

# 2. Auth guard
unauth = httpx.get("http://127.0.0.1:8000/api/model-info")
print("2. Auth Guard Status:", unauth.status_code)
assert unauth.status_code == 401

# 3. Model Info with auth
info = client.get("/api/model-info").json()
print("3. Model Info:", info["model"], "Quantization:", info["quantization"], "GPU:", info["gpu"]["name"])
assert info["model"] == "qwen2:7b"

# 4. Adaptive Chat - International Profile (English)
intl_req = {
    "prompt": "I am visiting Hanoi for 3 days. What temple etiquette should I follow?",
    "profile": {"market": "INTERNATIONAL", "language": "EN", "origin_country": "France", "destination": "Hà Nội"}
}
ans_intl = client.post("/api/chat", json=intl_req).json()
print("4. International Profile Reply Preview:", ans_intl["content"][:120] + "...")
assert ans_intl["status"] == "success"

# 5. Adaptive Chat - Local Profile (Vietnamese)
local_req = {
    "prompt": "Cuối tuần này ở Mộc Châu có điểm cắm trại nào vắng người?",
    "profile": {"travel_scope": "LOCAL", "persona": "FAMILY", "origin_province": "Sơn La", "destination": "Mộc Châu"}
}
ans_local = client.post("/api/chat", json=local_req).json()
print("5. Local Profile Reply Preview:", ans_local["content"][:120] + "...")
assert ans_local["status"] == "success"

# 6. Payload size limit guard (16KB max)
large_payload = {"prompt": "A" * 20000}
res_large = client.post("/api/chat", json=large_payload)
print("6. Large payload rejection status:", res_large.status_code)
assert res_large.status_code in [413, 422]

print("\nALL COLAB BACKEND VERIFICATION TESTS PASSED!")


In [ ]:
# STEP 7: AUTOMATED VERIFICATION SELF-TESTS
import httpx
headers = {"Authorization": f"Bearer {colab_app.COLAB_AI_API_TOKEN}"}
client = httpx.Client(base_url="http://127.0.0.1:8000", headers=headers, timeout=30.0)

# 1. Health check
h = client.get("/health").json()
print("1. Health Check:", h["status"], "GPU:", h["gpu"]["name"])
assert h["status"] == "healthy"

# 2. Auth guard
unauth = httpx.get("http://127.0.0.1:8000/api/model-info")
print("2. Auth Guard Status:", unauth.status_code)
assert unauth.status_code == 401

# 3. Model Info with auth
info = client.get("/api/model-info").json()
print("3. Model Info:", info["model"], "Quantization:", info["quantization"])

# 4. Adaptive Chat - International Profile
intl_req = {
    "prompt": "I am traveling to Hanoi. What temple etiquette should I follow?",
    "profile": {"customer_type": "INTERNATIONAL", "origin_country": "France", "destination": "Hà Nội"}
}
ans_intl = client.post("/api/chat", json=intl_req).json()
print("4. International Profile Reply Preview:", ans_intl["content"][:150] + "...")
assert ans_intl["personalized_for"] == "INTERNATIONAL"

# 5. Adaptive Chat - Local Profile
local_req = {
    "prompt": "Có quán ăn nào yên tĩnh ở Mộc Châu không?",
    "profile": {"customer_type": "LOCAL", "origin_province": "Sơn La", "destination": "Mộc Châu"}
}
ans_local = client.post("/api/chat", json=local_req).json()
print("5. Local Profile Reply Preview:", ans_local["content"][:150] + "...")
assert ans_local["personalized_for"] == "LOCAL"

print("\nALL COLAB BACKEND TESTS PASSED!")
